# 08 · Follow-up analysis: the class/example boundary, head bias, savings speed
### Machine Unlearning is Extinction, Not Forgetting

CPU-only. Three follow-ups to notebook 06:

| Part | Question | Status |
|---|---|---|
| **XH1-XH5** example-level relapse | Do unlearned *examples* come back under the same triggers that left unlearned *classes* hidden? | pre-registered 2026-10-01, before any example-level data |
| **B1** boundary | Is the share of forgetting that relapses larger at example level than at class level, per method? | pre-registered 2026-10-01 |
| **E1** head bias | How much of class-level forgetting is the output bias (cf. Zheng et al. 2026)? | exploratory |
| **E2** savings speed | Steps to reach 50 % forget accuracy when relearning, unlearned vs retrained (resolves the SCRUB discrepancy with the relearning-delay literature) | exploratory |

| Test | Statistic per unit (unlearned, matched; unit = seed × forget set) | Test |
|---|---|---|
| **XH1** spontaneous recovery | `mem_gap` after 5 epochs of retain fine-tuning (BN updated) minus before, **minus the same for the retrained model** | one-sided Wilcoxon, > 0 |
| **XH2** renewal | mean over the 4 contexts of (context − clean) `mem_gap`, minus the retrained model's | Wilcoxon, > 0 |
| **XH4** savings | area under the held-out `mem_gap` relearning curve, minus the retrained model's (mean over 100/500 shots) | Wilcoxon, > 0 |
| **XH5** leakage | 2 × \|MIA-AUROC − 0.5\| before any test, minus the retrained model's | Wilcoxon, > 0 |
| **B1** boundary | relapse fraction = (unlearned recovery − retrained recovery) / (original score − unlearned score), retain fine-tuning 5 epochs; example level (`mem_gap`) vs class level (`top5_cc`) | one-sided Mann-Whitney, example > class |
| **E1** head bias | forget accuracy after restoring the original bias; recall after one constant shift at 1 % FPR; each minus the retrained model's | Wilcoxon, > 0 |
| **E2** savings speed | steps to 50 % forget accuracy, retrained minus unlearned (positive = faster), mean over shot counts | Wilcoxon, > 0 |

Holm correction within each family (`xl-main`, `boundary`, `exploratory`).

## Kaggle settings for this notebook

Set these in the right-hand **Settings** panel (in the editor: *View → Show sidebar* if it is hidden).

| Setting | Value | Why |
|---|---|---|
| **Accelerator** | **None** | This notebook only does statistics and plots on the CPU; leaving the GPU off saves your weekly GPU quota. |
| **Internet** | **On** | Needed to download CIFAR-100 (~170 MB) and, in notebook 04, CIFAR-100-C, unless you attach them as datasets. Internet requires a phone-verified Kaggle account. |
| **Persistence** | Files only (optional) | Keeps `/kaggle/working` between *interactive* sessions, so you don't lose files if the browser disconnects. Not needed for background runs. |
| **Environment** | *Always use latest* for the first run, then **Pin to original environment** | Pinning freezes the library versions (PyTorch etc.), so a Kaggle image update can't break a half-finished experiment. |
| **Language** | Python | |

**Inputs to attach** (*Add Input* → *Your Work* / *Your Datasets*):

| Input | Why |
|---|---|
| `ext-code` (version 1.3) | The shared library. |
| `ext-01-originals` | `experiment_config.json` (checked automatically). |
| `ext-04-extinction` | Class-level relapse results: the class side of the boundary test and the steps-to-50% savings check. |
| `ext-07-example-level` | Example-level results, random design (run #9); only needed with `XL_DESIGN = "random"`. |
| `ext-09-atypical` + `ext-10-matched-search` | Example-level results, atypical design, and the head-bias check v2 (notebook 09). |

CIFAR-100 is **not** needed here (only result files are read); if the setup prints `CIFAR-100: NOT attached`, ignore it.

**How to run.** Click **Save Version → Save & Run All (Commit)**. The notebook then runs in the background
for up to 12 hours with the browser closed. When it finishes, its files appear under *Output*, and the next
notebook attaches them as an input. 

**If a run stops early** (time budget or error), nothing is lost. Every finished job is written to disk
immediately. Attach this notebook's own latest output as an input and run it again: finished jobs are
skipped and their files are carried over.

## Setup: import the shared library

This cell finds `ext_utils.py` and imports it as `U`. All the real code (data pipeline, model, unlearning
methods, tests, job runner) lives there, so every notebook uses *identical* code.

`U.env_report()` then prints:
- the Python and PyTorch versions,
- which GPUs are visible (you should see **two Tesla T4** with the T4 x2 accelerator),
- free disk space,
- whether the internet is reachable,
- which inputs are attached.

**Check this output before going further.** If it says `GPU: none found`, the accelerator is off. If it
says `internet: OFF` and no CIFAR-100 dataset is attached, data loading will fail.

In [ ]:
# Locate ext_utils.py (from an attached dataset, a utility script, or the working folder) and import it.
import os, sys, glob
candidates = (glob.glob("/kaggle/working/ext_utils.py")
              + glob.glob("/kaggle/usr/lib/**/ext_utils.py", recursive=True)
              + glob.glob("/kaggle/input/**/ext_utils.py", recursive=True)
              + glob.glob(os.path.join(os.getcwd(), "ext_utils.py"))
              + glob.glob(os.path.join(os.getcwd(), "..", "ext_utils.py")))
if not candidates:
    raise FileNotFoundError(
        "ext_utils.py not found. Upload it as a private Kaggle Dataset (e.g. 'ext-code') and attach it "
        "with 'Add Input' -> 'Your Datasets' (see the Kaggle settings section above).")
sys.path.insert(0, os.path.dirname(os.path.abspath(candidates[0])))
import ext_utils as U
print("ext_utils loaded from:", U.UTILS_FILE)
U.env_report()

## Configuration

Must match notebook 01 (checked). `MATCHED_ONLY` keeps matched unlearned models only, as in notebook 06.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, os, warnings
from scipy import stats
from statsmodels.stats.multitest import multipletests
warnings.filterwarnings("ignore")

CFG = U.make_cfg(
    smoke=False,
    norm="bn",
    seeds=[0, 1, 2],
    forget_classes=["maple_tree", "tiger", "rose", "shark", "bus",
                    "apple", "chair", "butterfly", "girl", "castle"],
    mctx_contexts=["clean", "gray", "blur"],
    xl_frac=0.1, xl_draws=[0, 1, 2], xl_pilot_draw=9,
)
U.check_experiment_config(CFG)
MATCHED_ONLY = True        # class level, as in notebook 06
XL_DESIGN = "atypical"     # which example-level arm: "atypical" (notebook 09, primary) or "random" (notebook 07, run #9)
XL_SCORE = "forget_acc" if XL_DESIGN == "atypical" else "mem_gap"   # pre-registered primary score per design
XL_MATCHED_ONLY = False    # pre-registered True for atypical, but 0/45 atypical models matched (run #12), as 0/45 random
                           # (run #9): all models are analysed as partial unlearning (deviation); see the partial-unlearning checks
XL_FORGET_GAP = 0.05       # notebook 09's matching gap on forget accuracy, for the forget-matched subset
ALPHA = 0.05
N_BOOT = 10000
os.makedirs(U.FIG_DIR, exist_ok=True)
TAB_DIR = os.path.join(U.WORK, "tables"); os.makedirs(TAB_DIR, exist_ok=True)
METHODS = CFG["methods"]

## Helpers

In [ ]:
RESULTS = []

def wilcoxon_greater(x):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    if len(x) < 2 or np.allclose(x, 0): return np.nan
    try: return float(stats.wilcoxon(x, alternative="greater").pvalue)
    except ValueError: return np.nan

def boot_ci(x, n=N_BOOT, seed=0):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    if len(x) == 0: return (np.nan, np.nan)
    means = np.random.default_rng(seed).choice(x, size=(n, len(x)), replace=True).mean(1)
    return tuple(np.percentile(means, [2.5, 97.5]))

def record(hyp, method, values, family, p=None, note=""):
    v = np.asarray(values, float); v = v[~np.isnan(v)]
    lo, hi = boot_ci(v)
    RESULTS.append(dict(family=family, hypothesis=hyp, method=method, n=len(v), mean=v.mean() if len(v) else np.nan,
                        ci_low=lo, ci_high=hi, p_raw=wilcoxon_greater(v) if p is None else p, note=note))

def minus_retrain(d, value, keys):
    """Value of each non-retrained model minus the retrained model's value for the same keys."""
    if d.empty: return pd.DataFrame(columns=list(d.columns) + ["excess"])
    ref_kind = "xlretrain" if (d["kind"] == "xlretrain").any() else "retrain"
    ref = d[d["kind"] == ref_kind][list(keys) + [value]].rename(columns={value: "ref"})
    out = d[d["kind"] != ref_kind].merge(ref, on=list(keys), how="inner")
    return out.assign(excess=out[value] - out["ref"])

def not_collapsed(d):
    if "collapsed" not in d: return d
    return d[~d["collapsed"].astype("boolean").fillna(False).astype(bool)]

## Load results

Example-level rows: the pilot forget set is dropped, unlearned models are filtered to matched ones, and collapsed
fine-tuning rows are dropped. Class-level rows (notebook 04) are filtered as in notebook 06.

In [ ]:
xu = U.load_df("xl_unlearn", CFG)
xe = U.load_df("xl_extinction", CFG)
hb = U.load_df("headbias", CFG)
ce = U.load_df("extinction", CFG)
for name, d in [("xl_unlearn", xu), ("xl_extinction", xe), ("headbias", hb), ("extinction (class)", ce)]:
    print(f"{name:20s}: {len(d)} rows")
XID = ["kind", "method", "seed", "draw"]
CID = ["kind", "method", "cls", "seed"]

def by_design(d):
    if d.empty: return d
    des = d["design"].fillna("random") if "design" in d else pd.Series("random", index=d.index)
    return d[des == XL_DESIGN]

def keep_xl(d):
    if d.empty: return d
    d = d[d["draw"].isin(CFG["xl_draws"]) & d["seed"].isin(CFG["seeds"])]
    if XL_MATCHED_ONLY and "matched" in d:
        d = d[(d["kind"] != "xlunl") | (d["matched"] == True)]
    return not_collapsed(d)

def keep_class(d):
    if d.empty: return d
    d = d[d["cls"] != CFG["pilot_class"]]
    if MATCHED_ONLY and "matched" in d:
        d = d[~d["kind"].isin(["unl", "aba", "mctx"]) | (d["matched"] == True)]
    return not_collapsed(d)

xu, xe = by_design(xu), by_design(xe)
# Matched-model search (notebook 10, ext_utils 1.5): a variant such as "RL-long" runs its base method with a longer
# schedule. Rule fixed before notebook 10's data: per (method, seed, forget set) a matched variant model replaces
# notebook 09's model; otherwise notebook 09's model stays. XV / XEV keep every variant for descriptive comparisons.
XV, XEV = pd.DataFrame(), pd.DataFrame()
if len(xu):
    xu = xu.assign(variant=xu["method"], method=xu["method"].astype(str).str.split("-").str[0])
    xu["matched"] = xu["matched"].astype("boolean").fillna(False).astype(bool)
    XV = xu.copy()
    prio = np.where((xu["variant"] != xu["method"]) & xu["matched"], 2, np.where(xu["variant"] == xu["method"], 1, 0))
    xu = xu.assign(_prio=prio).sort_values("_prio").groupby(["method", "seed", "draw"]).tail(1).drop(columns="_prio")
    used = xu[xu["draw"].isin(CFG["xl_draws"])].groupby(["method", "variant"])["matched"].agg(["count", "sum"])
    print("example-level models used (count, matched):"); display(used)
if len(xe):
    xe = xe.assign(variant=xe["method"], method=xe["method"].astype(str).str.split("-").str[0])
    XEV = keep_xl(xe.copy())
    if len(xu):
        keep = set(zip(xu["variant"], xu["seed"], xu["draw"]))
        inkeep = pd.Series([k in keep for k in zip(xe["variant"], xe["seed"], xe["draw"])], index=xe.index)
        xe = xe[(xe["kind"] != "xlunl") | inkeep]
xe, ce = keep_xl(xe), keep_class(ce)
print(f"example-level design: {XL_DESIGN} | primary score: {XL_SCORE} | matched only: {XL_MATCHED_ONLY}")
if len(hb):
    hb = hb[hb["cls"] != CFG["pilot_class"]]
    if MATCHED_ONLY: hb = hb[~hb["kind"].isin(["unl", "aba", "mctx"]) | (hb["matched"] == True)]
if len(xu):
    xm = xu[xu["draw"].isin(CFG["xl_draws"])]
    print("example-level matched rate:", xm.groupby("method")["matched"].mean().round(3).to_dict())
    display(xm.groupby("method")[["forget_acc", "ref_forget_acc", "test_acc", "ref_test_acc", "mem_gap", "ref_mem_gap", "leak"]].mean().round(4))
    if len(xe):
        ob = xe[(xe["test"] == "base") & (xe["kind"] == "original")][["seed", "draw", "forget_acc"]].rename(columns={"forget_acc": "orig_forget_acc"})
        share = xm.merge(ob, on=["seed", "draw"])
        den = share["orig_forget_acc"] - share["ref_forget_acc"]
        share["share_forgotten"] = np.where(den > 0.02, (share["orig_forget_acc"] - share["forget_acc"]) / den, np.nan)
        print("share of the original-to-retrained forgetting gap each method achieved (1 = as much as retraining):")
        display(share.groupby("method")["share_forgotten"].agg(["mean", "min", "max"]).round(3))

## XH1: spontaneous recovery at example level

Recovery = the primary score (`forget_acc` for the atypical design, `mem_gap` for the random one) after the manipulation minus before. Primary: 5 epochs of retain fine-tuning, scored with the
BatchNorm statistics the fine-tuning produced; tested as unlearned minus retrained. All proxies are tabulated.

In [ ]:
xsr = pd.DataFrame()
if len(xe) and (xe["test"] == "spontaneous").any():
    xbase = xe[xe["test"] == "base"][XID + [XL_SCORE]].rename(columns={XL_SCORE: "base"})
    xsr = xe[xe["test"] == "spontaneous"].merge(xbase, on=XID)
    xsr["recovery"] = xsr[XL_SCORE] - xsr["base"]
    prim = xsr[(xsr["proxy"] == "retain_ft") & (xsr["bn"] == "update") & np.isclose(xsr["level"].astype(float), CFG["sr_epochs"])]
    e = minus_retrain(prim, "recovery", ["seed", "draw"])
    for m in METHODS:
        record("XH1 spontaneous recovery (example level)", m, e[(e.kind == "xlunl") & (e.method == m)]["excess"], "xl-main")
    fr = xsr[(xsr["proxy"] == "retain_ft") & (xsr["bn"] == "frozen") & np.isclose(xsr["level"].astype(float), CFG["sr_epochs"])]
    ef = minus_retrain(fr, "recovery", ["seed", "draw"])
    for m in METHODS:
        record("R: XH1 with original BN statistics", m, ef[(ef.kind == "xlunl") & (ef.method == m)]["excess"], "xl-robustness")
    t = xsr[xsr.kind.isin(["xlunl", "xlretrain", "original"])].pivot_table(index=["proxy", "bn", "level"], columns="method", values="recovery", aggfunc="mean")
    t.to_csv(os.path.join(TAB_DIR, "XH1_spontaneous_all_proxies.csv")); display(t.round(3))
else:
    print("No example-level results yet (run notebook 07).")

## XH2: renewal at example level

In [ ]:
if len(xe) and (xe["test"] == "context").any():
    xc = xe[xe["test"] == "context"].merge(xe[xe["test"] == "base"][XID + [XL_SCORE]].rename(columns={XL_SCORE: "base"}), on=XID)
    xc["renewal"] = xc[XL_SCORE] - xc["base"]
    per = xc.groupby(XID)["renewal"].mean().reset_index()
    e = minus_retrain(per, "renewal", ["seed", "draw"])
    for m in METHODS:
        record("XH2 renewal (example level)", m, e[(e.kind == "xlunl") & (e.method == m)]["excess"], "xl-main")
    display(xc.pivot_table(index="context", columns="method", values="renewal", aggfunc="mean").round(3))

## XH4: savings at example level

Relearn from 100 or 500 forgotten images; score the other forgotten images. Area under the held-out `mem_gap`
curve, minus the retrained model's, averaged over the two shot counts per unit.

In [ ]:
_trapz = getattr(np, "trapezoid", None) or np.trapz
def auc_curve(g, col):
    g = g.sort_values("level"); x = g["level"].astype(float).values; y = g[col].values
    return float(_trapz(y, x) / (x.max() - x.min())) if len(x) > 1 and x.max() > x.min() else float(y.mean())

if len(xe) and (xe["test"] == "savings").any():
    sv = xe[(xe["test"] == "savings") & (xe["bn"] == "update")]
    a = pd.DataFrame([dict(zip(XID + ["shots"], k), auc=auc_curve(g, XL_SCORE)) for k, g in sv.groupby(XID + ["shots"])])
    e = minus_retrain(a, "auc", ["seed", "draw", "shots"])
    per = e.groupby(XID)["excess"].mean().reset_index()
    for m in METHODS:
        record("XH4 savings (example level)", m, per[(per.kind == "xlunl") & (per.method == m)]["excess"], "xl-main")
    display(e.pivot_table(index="shots", columns="method", values="excess", aggfunc="mean").round(3))
    # Robustness (added after run #13): the AUC includes the starting level, so models that forgot little look fast.
    # Gain over the model's own score at step 0 removes that.
    def gain_auc(g):
        g = g.sort_values("level"); return auc_curve(g.assign(_g=g[XL_SCORE] - g[XL_SCORE].iloc[0]), "_g")
    ag = pd.DataFrame([dict(zip(XID + ["shots"], k), auc=gain_auc(g)) for k, g in sv.groupby(XID + ["shots"])])
    eg = minus_retrain(ag, "auc", ["seed", "draw", "shots"]).groupby(XID)["excess"].mean().reset_index()
    for m in METHODS:
        record("R: XH4 savings, gain over own start", m, eg[(eg.kind == "xlunl") & (eg.method == m)]["excess"], "xl-robustness")

## XH5: membership leakage before any test

In [ ]:
if len(xe):
    b = xe[xe["test"] == "base"]
    e = minus_retrain(b, "leak", ["seed", "draw"])
    for m in METHODS:
        record("XH5 leakage (example level)", m, e[(e.kind == "xlunl") & (e.method == m)]["excess"], "xl-main")
    display(b.groupby(["kind", "method"])[["mia_auc", "leak", "mem_gap"]].mean().round(3))
    # Robustness (added after run #13): atypical images are harder than test images, so even the retrained model
    # separates them (AUROC < 0.5) and |AUROC - 0.5| rewards under-forgetting. Signed: unlearned minus retrained AUROC
    # (> 0 = the model is more confident on its forgotten images than a model that never saw them).
    es = minus_retrain(b, "mia_auc", ["seed", "draw"])
    for m in METHODS:
        record("R: XH5 MIA AUROC minus retrained (signed)", m, es[(es.kind == "xlunl") & (es.method == m)]["excess"], "xl-robustness")

## B1: the boundary between class-level and example-level forgetting

**Relapse fraction** = (unlearned recovery − retrained recovery) / (original score − unlearned score before), for
5 epochs of retain fine-tuning (BN updated). It is the share of the forgetting that comes back, beyond what a
never-trained model gains: 0 = none, 1 = all. Class level uses `top5_cc` (notebook 04), example level `mem_gap`.
Units whose forgetting gap is below 0.05 are left out (the fraction is undefined there).

In [ ]:
def relapse_fraction(d, score, ids, keys, unl_kind, ref_kind):
    base = d[d["test"] == "base"][ids + [score]].rename(columns={score: "before"})
    rf = d[(d["test"] == "spontaneous") & (d["proxy"] == "retain_ft") & (d["bn"] == "update")
           & np.isclose(d["level"].astype(float), CFG["sr_epochs"])].merge(base, on=ids)
    rf["delta"] = rf[score] - rf["before"]
    ref = rf[rf.kind == ref_kind][keys + ["delta"]].rename(columns={"delta": "delta_ref"})
    orig = base[base.kind == "original"][keys + ["before"]].rename(columns={"before": "s_orig"})
    u = rf[rf.kind == unl_kind].merge(ref, on=keys).merge(orig, on=keys)
    gap = u["s_orig"] - u["before"]
    u["relapse_fraction"] = np.where(gap > 0.05, (u["delta"] - u["delta_ref"]) / gap, np.nan)
    return u

bnd = []
if len(ce):
    uc = relapse_fraction(ce, "top5_cc", CID, ["cls", "seed"], "unl", "retrain"); uc["level_of_forgetting"] = "class"; bnd.append(uc)
if len(xe):
    ux = relapse_fraction(xe, XL_SCORE, XID, ["seed", "draw"], "xlunl", "xlretrain"); ux["level_of_forgetting"] = "example"; bnd.append(ux)
if len(bnd) == 2:
    B = pd.concat(bnd, ignore_index=True)
    for m in METHODS:
        x = B[(B.method == m) & (B.level_of_forgetting == "example")]["relapse_fraction"].dropna()
        c = B[(B.method == m) & (B.level_of_forgetting == "class")]["relapse_fraction"].dropna()
        p = float(stats.mannwhitneyu(x, c, alternative="greater").pvalue) if len(x) > 1 and len(c) > 1 else np.nan
        RESULTS.append(dict(family="boundary", hypothesis="B1 relapse fraction, example minus class", method=m,
                            n=len(x) + len(c), mean=x.mean() - c.mean(), ci_low=np.nan, ci_high=np.nan, p_raw=p,
                            note=f"example {x.mean():.3f} (n={len(x)}), class {c.mean():.3f} (n={len(c)})"))
    display(B.groupby(["level_of_forgetting", "method"])["relapse_fraction"].agg(["mean", "median", "count"]).round(3))
else:
    B = pd.DataFrame()
    print("Boundary test needs both notebook 04 (class) and notebook 07 (example) results.")

## Is the relapse just partial unlearning? (exploratory, added after run #12)

No atypical model met the matching rule (run #12): RL, SalUn, FT and SCRUB stopped short of the retrained model's
forget accuracy, and NegGrad+ reached it only by losing ~10 test points. Three checks:
1. **Level after retain fine-tuning vs the retrained model's level after the same fine-tuning.** A model that never saw
   these images gets ~3 % of them right at full test accuracy, so anything far above that was brought back from a trace,
   not rebuilt from general features (this also covers NegGrad+ regaining test accuracy).
2. **Forget-matched models only:** forget accuracy within ±5 points of the retrained model's, test accuracy ignored.
3. **Dose-response:** if relapse were only left-over knowledge, models that forgot more should relapse less
   (Spearman correlation of the forget accuracy before the test with the recovery; FT excluded, because its
   unlearning *is* retain fine-tuning, so the probe only continues it).


In [ ]:
if len(xsr) and len(xu):
    p5 = xsr[(xsr["proxy"] == "retain_ft") & (xsr["bn"] == "update") & np.isclose(xsr["level"].astype(float), CFG["sr_epochs"])]
    lv = p5.groupby(["kind", "method"])[["base", XL_SCORE, "recovery"]].mean().rename(columns={"base": "before", XL_SCORE: "after retain FT"})
    print(f"{XL_SCORE} before and after {CFG['sr_epochs']} epochs of retain fine-tuning (BN update), means:")
    display(lv.round(3))
    ra = p5[p5.kind == "xlretrain"][["seed", "draw", XL_SCORE]].rename(columns={XL_SCORE: "retrain_after"})
    info = xu[["method", "seed", "draw", "forget_acc", "ref_forget_acc", "test_acc"]].rename(
        columns={"forget_acc": "unl_forget_acc", "test_acc": "unl_test_acc"})
    u = p5[p5.kind == "xlunl"].merge(ra, on=["seed", "draw"]).merge(info, on=["method", "seed", "draw"])
    u["above_retrain"] = u[XL_SCORE] - u["retrain_after"]
    u["forget_matched"] = (u["unl_forget_acc"] - u["ref_forget_acc"]).abs() <= XL_FORGET_GAP
    fm = minus_retrain(p5[p5.kind.isin(["xlunl", "xlretrain"])], "recovery", ["seed", "draw"])
    fm = fm.merge(u[["method", "seed", "draw", "forget_matched"]], on=["method", "seed", "draw"])
    for m in METHODS:
        record("R: level after retain FT minus retrained level", m, u[u.method == m]["above_retrain"], "xl-robustness")
        x = fm[(fm.method == m) & fm.forget_matched]["excess"]
        if len(x): record("R: XH1, forget-matched models only (test accuracy ignored)", m, x, "xl-robustness")
    print("forget-matched models per method:", u.groupby("method")["forget_matched"].sum().astype(int).to_dict())
    dr = u[u.method != "FT"].dropna(subset=["base", "recovery"])
    rows = []
    for m, g in [("pooled (no FT)", dr)] + list(dr.groupby("method")):
        rho, p = stats.spearmanr(g["base"], g["recovery"]) if len(g) > 2 and g["base"].nunique() > 1 else (np.nan, np.nan)
        rows.append(dict(models=m, n=len(g), spearman_rho=rho, p=p))
    print("dose-response: forget accuracy before the test vs recovery (positive = less-forgotten models relapse more):")
    display(pd.DataFrame(rows).round(3))
    u.to_csv(os.path.join(TAB_DIR, "XH1_partial_unlearning_checks.csv"), index=False)


## Matched models only, and the longer schedules (notebook 10)

XH1 and B1 restricted to models that met the matching rule (pre-registered analysis, now possible for the methods that
matched). Then every variant side by side (notebook 09's schedule vs notebook 10's longer one), matched or not: does
relapse shrink when forgetting is more complete?


In [ ]:
if len(xsr) and len(xu):
    mk = xu[xu["matched"] & (xu["kind"] == "xlunl") & xu["draw"].isin(CFG["xl_draws"])][["method", "seed", "draw"]]
    print("matched models per method:", mk.groupby("method").size().to_dict())
    p5 = xsr[(xsr["proxy"] == "retain_ft") & (xsr["bn"] == "update") & np.isclose(xsr["level"].astype(float), CFG["sr_epochs"])]
    em = minus_retrain(p5[p5.kind.isin(["xlunl", "xlretrain"])], "recovery", ["seed", "draw"]).merge(mk, on=["method", "seed", "draw"])
    for m in METHODS:
        x = em[em.method == m]["excess"]
        if len(x): record("XH1, matched models only", m, x, "xl-matched")
    if len(B):
        bx = B[B.level_of_forgetting == "example"].merge(mk, on=["method", "seed", "draw"])
        for m in METHODS:
            x = bx[bx.method == m]["relapse_fraction"].dropna()
            c = B[(B.method == m) & (B.level_of_forgetting == "class")]["relapse_fraction"].dropna()
            if len(x):
                p = float(stats.mannwhitneyu(x, c, alternative="greater").pvalue) if len(x) > 1 and len(c) > 1 else np.nan
                RESULTS.append(dict(family="xl-matched", hypothesis="B1, matched models only", method=m, n=len(x) + len(c),
                                    mean=x.mean() - c.mean(), ci_low=np.nan, ci_high=np.nan, p_raw=p,
                                    note=f"example {x.mean():.3f} (n={len(x)}), class {c.mean():.3f} (n={len(c)})"))
if len(XEV) and len(XV):
    v5 = XEV[(XEV["test"] == "spontaneous") & (XEV["proxy"] == "retain_ft") & (XEV["bn"] == "update")
             & np.isclose(pd.to_numeric(XEV["level"], errors="coerce"), CFG["sr_epochs"])]
    vb = XEV[XEV["test"] == "base"][["kind", "variant", "seed", "draw", XL_SCORE]].rename(columns={XL_SCORE: "before"})
    v5 = v5.merge(vb, on=["kind", "variant", "seed", "draw"]).assign(recovery=lambda d: d[XL_SCORE] - d["before"])
    vr = v5[v5.kind == "xlretrain"][["seed", "draw", "recovery", XL_SCORE]].rename(columns={"recovery": "ref_rec", XL_SCORE: "ref_after"})
    vu = v5[v5.kind == "xlunl"].merge(vr, on=["seed", "draw"]).merge(
        XV[["variant", "seed", "draw", "matched", "test_acc"]].rename(columns={"matched": "is_matched", "test_acc": "unl_test_acc"}),
        on=["variant", "seed", "draw"])
    vu["excess"] = vu["recovery"] - vu["ref_rec"]
    vu["above_retrain"] = vu[XL_SCORE] - vu["ref_after"]
    print(f"every variant: {XL_SCORE} before / after retain FT, recovery minus retrained, matched share:")
    display(vu.groupby(["method", "variant"]).agg(n=("excess", "size"), matched=("is_matched", "mean"),
                                                  test_acc=("unl_test_acc", "mean"), before=("before", "mean"),
                                                  after=(XL_SCORE, "mean"), excess=("excess", "mean"),
                                                  above_retrain=("above_retrain", "mean")).round(3))
    for v in sorted(set(vu["variant"]) - set(METHODS)):
        record("R: XH1, notebook 10 schedule (all models)", v, vu[vu.variant == v]["excess"], "xl-robustness")
    vu.to_csv(os.path.join(TAB_DIR, "XH1_by_variant.csv"), index=False)


## E1 (exploratory): head bias at class level

Each value is the unlearned model's minus the retrained model's (same class and seed). The retrained model has
never learned the class, so its restored-bias accuracy and shifted recall are the floor.

In [ ]:
if len(hb):
    cols = ["bias_z", "bias_z_orig", "row_cos", "row_norm_ratio", "bias_restore_forget_acc", "row_restore_forget_acc", "shift_recall", "shift_fpr"]
    cols += [c for c in ["shift_recall_fpr1", "shift_fpr_fpr1", "shift_recall_fpr5", "shift_fpr_fpr5"] if c in hb]
    SHIFT = "shift_recall_fpr1" if "shift_recall_fpr1" in hb else "shift_recall"
    display(hb.groupby(["kind", "method"])[cols].mean().round(3))
    for val, name in [("bias_restore_forget_acc", "E1a forget accuracy after restoring the original bias"),
                      (SHIFT, "E1b recall after one constant shift (1% FPR)")]:
        e = minus_retrain(hb[hb.kind.isin(["unl", "retrain"])], val, ["cls", "seed"])
        for m in METHODS:
            record(name, m, e[(e.kind == "unl") & (e.method == m)]["excess"], "exploratory")
    hb.groupby(["kind", "method"])[cols].mean().to_csv(os.path.join(TAB_DIR, "E1_headbias.csv"))
else:
    print("No head-bias results yet (notebook 07, stage H).")

## E2 (exploratory): savings speed at class level

Steps until forget-class accuracy first reaches 50 % during relearning (notebook 04, BN updated; 200 = not
reached within 100 steps). Positive values = the unlearned model gets there faster than the retrained one.

In [ ]:
def steps_to(g, thr=0.5, never=200):
    g = g.sort_values("level"); hit = g[g["forget_test_acc"] >= thr]
    return float(hit["level"].iloc[0]) if len(hit) else float(never)

S = pd.DataFrame()
if len(ce) and (ce["test"] == "savings").any():
    sv = ce[(ce["test"] == "savings") & (ce["bn"] == "update") & ce["kind"].isin(["unl", "retrain"])]
    S = pd.DataFrame([dict(zip(CID + ["shots"], k), steps=steps_to(g)) for k, g in sv.groupby(CID + ["shots"])])
    ref = S[S.kind == "retrain"][["cls", "seed", "shots", "steps"]].rename(columns={"steps": "ref_steps"})
    S2 = S[S.kind == "unl"].merge(ref, on=["cls", "seed", "shots"]).assign(faster=lambda d: d["ref_steps"] - d["steps"])
    per = S2.groupby(CID)["faster"].mean().reset_index()
    for m in METHODS:
        record("E2 steps to 50% (retrained minus unlearned)", m, per[per.method == m]["faster"], "exploratory")
    display(S.pivot_table(index="shots", columns="method", values="steps", aggfunc="median").round(1))

## All tests with Holm correction

In [ ]:
res = pd.DataFrame(RESULTS)
if len(res):
    res["p_holm"] = np.nan
    for fam, idx in res.groupby("family").groups.items():
        sub = res.loc[idx]; ok = sub["p_raw"].notna()
        if ok.sum(): res.loc[sub.index[ok], "p_holm"] = multipletests(sub.loc[ok, "p_raw"], method="holm")[1]
    res["significant"] = res["p_holm"] < ALPHA
    res.to_csv(os.path.join(TAB_DIR, "followup_tests.csv"), index=False)
    display(res.round(4))
else:
    print("No results to test yet.")

## Figures

9. **Boundary**: relapse fraction, class vs example level, per method (the paper's key contrast).
10. **Example-level retain fine-tuning**: `mem_gap` recovery per epoch, unlearned vs retrained.
11. **Head bias**: forget accuracy after restoring the bias, and recall after one constant shift.
12. **Savings speed**: median steps to 50 % per method and shot count.

In [ ]:
def savefig(name):
    plt.tight_layout()
    for ext_ in ("png", "pdf"):
        plt.savefig(os.path.join(U.FIG_DIR, f"{name}.{ext_}"), dpi=300, bbox_inches="tight")
    plt.show()

if len(B):
    g = B.groupby(["method", "level_of_forgetting"])["relapse_fraction"]
    mean, lo, hi = g.mean().unstack(), g.apply(lambda v: boot_ci(v)[0]).unstack(), g.apply(lambda v: boot_ci(v)[1]).unstack()
    fig, ax = plt.subplots(figsize=(6, 3.5)); xs = np.arange(len(mean.index)); w = 0.38
    for j, lvl in enumerate(["class", "example"]):
        if lvl in mean:
            ax.bar(xs + (j - 0.5) * w, mean[lvl], w, yerr=[mean[lvl] - lo[lvl], hi[lvl] - mean[lvl]], capsize=3, label=f"{lvl}-level forgetting")
    ax.axhline(0, color="grey", lw=0.8); ax.set_xticks(xs); ax.set_xticklabels(mean.index)
    ax.set_ylabel("share of forgetting that relapses"); ax.set_title("Retain fine-tuning brings back examples, not classes"); ax.legend(frameon=False, fontsize=8)
    savefig("fig9_boundary")

if len(xsr):
    d0 = xsr[(xsr.proxy == "retain_ft") & (xsr.bn == "update") & xsr.kind.isin(["xlunl", "xlretrain"])]
    fig, ax = plt.subplots(figsize=(5, 3.5))
    for m, gg in d0.groupby("method"):
        gg = gg.groupby("level")["recovery"].mean()
        ax.plot(gg.index, gg.values, marker="o", label=m, color="black" if m == "retrain" else None, ls="--" if m == "retrain" else "-")
    ax.axhline(0, color="grey", lw=0.8); ax.set_xlabel("epochs of retain fine-tuning"); ax.set_ylabel(f"{XL_SCORE} recovery")
    ax.set_title("Example level: retain fine-tuning"); ax.legend(frameon=False, fontsize=7)
    savefig("fig10_xl_retain_ft")

if len(hb):
    t = hb[hb.kind.isin(["unl", "retrain"])].groupby("method")[["bias_restore_forget_acc", "row_restore_forget_acc", SHIFT]].mean()
    t = t.rename(columns={"bias_restore_forget_acc": "restore bias (1 number)", "row_restore_forget_acc": "restore output row", SHIFT: "one constant shift (1% FPR)"})
    ax = t.plot(kind="bar", figsize=(6.5, 3.5), width=0.8); ax.set_ylabel("forget-class accuracy / recall"); ax.set_xlabel("")
    ax.set_title("How much of class forgetting sits in the output layer"); ax.legend(frameon=False, fontsize=7)
    savefig("fig11_headbias")

if len(S):
    t = S.pivot_table(index="shots", columns="method", values="steps", aggfunc="median")
    ax = t.T.plot(kind="bar", figsize=(6, 3.5), width=0.8); ax.set_ylabel("median steps to 50% (200 = never)"); ax.set_xlabel("")
    ax.set_title("Relearning speed (class level)"); ax.legend(title="images", frameon=False, fontsize=7)
    savefig("fig12_steps_to_50")

## What to report

- `tables/followup_tests.csv`: every follow-up test with family, n, mean, bootstrap CI, raw and Holm p.
- **XH1-XH5 and B1 were fixed on 2026-10-01, before any example-level data existed**; state that in the paper.
  E1 and E2 are exploratory and must be labelled so.
- The boundary claim needs **XH1 positive and B1 significant**: example-level forgetting relapses under retain
  fine-tuning while class-level forgetting does not.

## Name of this run's output

Title this notebook **`ext-08-followup`** and save its output as a dataset of the same name (tables and figures).